# N30B - Benchmark cuantitativo del RAG Agent sobre el reglamento FIA

> Cuaderno hermano de `N30_rag_agent.ipynb`. El cuaderno original conserva la
> evaluación cualitativa con cuatro queries de demostración; este cuaderno
> añade la evaluación **cuantitativa** que la memoria del TFG (capítulo 5)
> necesita para reportar cifras citables.

## Objetivo

Construir un set de evaluación con **15 queries en español** sobre el
reglamento deportivo de la FIA (temporadas 2023-2025), con ground truth
manual verificada artículo a artículo contra los PDFs originales, y medir
**Precision@k**, **MRR** y **latencia** para tres configuraciones del
retriever:

1. `fia_regulations` - producción (BGE-M3 1024d, chunk 512, overlap 64).
2. `fia_regulations_minilm_v1` - baseline alternativo de embedding
   (MiniLM-L6-v2 384d, chunk 512, overlap 64).
3. `fia_regulations_bge_chunk256_v1` - variante de granularidad de chunk
   (BGE-M3 1024d, chunk 256, overlap 32).

Todas las colecciones viven en el mismo `data/rag/qdrant_local/`. La
colección de producción **no se modifica**: solo se añaden las dos nuevas y
se consultan en modo lectura. La construcción del set de queries respeta la
hard rule de **no inventar ground truth** - cada artículo y cada keyword se
verifica como substring literal del PDF correspondiente.

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer


C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Localización de la raíz del repositorio

Walker estándar (mismo patrón que `N31_strategy_orchestrator.ipynb`) para
poder ejecutar el cuaderno desde cualquier subcarpeta sin hardcodear la
ruta absoluta. Inserta `repo_root` en `sys.path` por si se necesita
importar desde `src/` o `scripts/`.

In [2]:
def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / ".git").exists():
            return parent
    raise RuntimeError("Could not locate repo root from current directory")


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo_root = {REPO_ROOT}")


repo_root = C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager


## Helpers de chunking reutilizados

Reusamos las funciones puras de `scripts/build_rag_index.py` para chunkear
los PDFs. Esto evita duplicar código y garantiza que las colecciones nuevas
se construyen con el mismo pipeline que la de producción - solo cambian los
parámetros (`chunk_size`, `chunk_overlap`) y el modelo de embedding.

In [3]:
from scripts.build_rag_index import (  # noqa: E402
    PDFDocument,
    TextChunk,
    clean_text,
    compute_hash,
    embed_chunks,
    ensure_collection,
    extract_article_reference,
    extract_section_title,
    iter_chunks,
    load_pdf_documents,
    parse_pdf_filename,
    upsert_chunks,
)


## Carga del set de evaluación

`queries_v1.json` contiene 15 queries en español repartidas en cinco
categorías (`tyre_allocation`, `pit_stops`, `safety_car`,
`flags_penalties`, `drs`) y tres temporadas (2023-2025). Cada entrada lleva
el artículo exacto que la responde, una sección, una lista de keywords
literales del PDF y una racional en inglés.

In [4]:
QUERIES_PATH = REPO_ROOT / "data" / "rag_eval" / "queries_v1.json"
QUERIES = json.loads(QUERIES_PATH.read_text(encoding="utf-8"))

queries_df = pd.DataFrame(
    [
        {
            "id": q["id"],
            "year": q["year"],
            "category": q["category"],
            "article": q["ground_truth"]["article"],
            "query": q["query"],
        }
        for q in QUERIES
    ]
)

print(f"Loaded {len(QUERIES)} queries from {QUERIES_PATH}")
print(queries_df.groupby("category").size().rename("n_queries"))
print()
print(queries_df.groupby("year").size().rename("n_queries"))
queries_df


Loaded 15 queries from C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\rag_eval\queries_v1.json
category
drs                2
flags_penalties    3
pit_stops          3
safety_car         3
tyre_allocation    4
Name: n_queries, dtype: int64

year
2023     4
2025    11
Name: n_queries, dtype: int64


,id,year,category,article,query
0,Q01,2025,tyre_allocation,30.2,¿Cuántos juegos de neumáticos slick puede usar...
1,Q02,2023,tyre_allocation,30.2,¿Cuántos juegos de neumáticos intermedios y de...
2,Q03,2025,tyre_allocation,30.2,¿Cuál es el compuesto obligatorio para Q3 segú...
3,Q04,2025,tyre_allocation,30.5,¿Está obligado un piloto a usar al menos dos c...
4,Q05,2025,pit_stops,34.7,¿Cuál es el límite de velocidad en el pit lane...
5,Q06,2025,pit_stops,34.14,¿Qué sanción aplica si un coche es liberado de...
6,Q07,2025,pit_stops,56.4,¿Puede un coche entrar al pit lane bajo Virtua...
7,Q08,2025,safety_car,55.4,¿Qué mensaje envía la FIA por el sistema ofici...
8,Q09,2025,safety_car,56.2,¿Qué señal indica el inicio del procedimiento ...
9,Q10,2023,safety_car,55.4,¿Qué se comunica a los Competidores cuando el ...


## Construcción idempotente de las colecciones nuevas

Abrimos un único `QdrantClient` sobre `data/rag/qdrant_local/`. La colección
`fia_regulations` ya está poblada por `scripts/build_rag_index.py` y no la
tocamos. Las dos nuevas (`fia_regulations_minilm_v1` y
`fia_regulations_bge_chunk256_v1`) se crean solo si no existen aún -
ejecutar Restart → Run All una segunda vez es seguro y rápido.

In [5]:
QDRANT_PATH = REPO_ROOT / "data" / "rag" / "qdrant_local"
DOCS_DIR = REPO_ROOT / "data" / "rag" / "documents"

PROD_COLLECTION = "fia_regulations"
MINILM_COLLECTION = "fia_regulations_minilm_v1"
BGE_CHUNK256_COLLECTION = "fia_regulations_bge_chunk256_v1"

PROD_MODEL = "BAAI/bge-m3"
MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

client = QdrantClient(path=str(QDRANT_PATH))
existing_collections = {c.name for c in client.get_collections().collections}
print("Existing Qdrant collections:", sorted(existing_collections))


Existing Qdrant collections: ['fia_regulations', 'fia_regulations_bge_chunk256_v1', 'fia_regulations_minilm_v1']


In [6]:
def _build_collection(
    client: QdrantClient,
    *,
    name: str,
    encoder: SentenceTransformer,
    vector_size: int,
    chunk_size: int,
    chunk_overlap: int,
    docs_dir: Path,
) -> int:
    """Build a Qdrant collection from the FIA PDFs with the given parameters.

    Idempotent: if the collection already exists and is populated, returns the
    current point count without re-embedding. Otherwise it loads every PDF,
    re-chunks with `(chunk_size, chunk_overlap)`, embeds the chunks with
    `encoder`, and upserts them into the named collection.
    """
    ensure_collection(client, name, vector_size)
    info = client.get_collection(name)
    current_points = info.points_count or 0
    if current_points > 0:
        print(f"  -> '{name}' already populated with {current_points} chunks; skipping rebuild.")
        return current_points

    documents = load_pdf_documents(docs_dir)
    if not documents:
        raise RuntimeError(f"No PDFs found in {docs_dir}")

    all_chunks: list[TextChunk] = []
    for doc in documents:
        for chunk in iter_chunks(doc, chunk_size=chunk_size, chunk_overlap=chunk_overlap):
            all_chunks.append(chunk)

    print(f"  -> embedding {len(all_chunks)} chunks for '{name}' with {encoder._first_module().__class__.__name__}...")
    embeddings = embed_chunks(all_chunks, encoder)
    n_upserted = upsert_chunks(client, name, all_chunks, embeddings, id_offset=0)
    print(f"  -> upserted {n_upserted} chunks into '{name}'")
    return n_upserted


In [7]:
# Production collection (read-only) - sanity check
prod_info = client.get_collection(PROD_COLLECTION)
print(f"Production '{PROD_COLLECTION}': {prod_info.points_count} points (read-only)")


Production 'fia_regulations': 2279 points (read-only)


In [8]:
# Build MiniLM collection (chunk 512 / overlap 64, 384-dim)
print(f"\nBuilding {MINILM_COLLECTION}...")
minilm_encoder = SentenceTransformer(MINILM_MODEL)
n_minilm = _build_collection(
    client,
    name=MINILM_COLLECTION,
    encoder=minilm_encoder,
    vector_size=384,
    chunk_size=512,
    chunk_overlap=64,
    docs_dir=DOCS_DIR,
)
print(f"{MINILM_COLLECTION} ready ({n_minilm} points)")


18:29:27  INFO      Use pytorch device_name: cuda:0


18:29:27  INFO      Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2



Building fia_regulations_minilm_v1...


18:29:27  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


18:29:27  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"


...

18:29:29  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


18:29:29  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


18:29:29  INFO      HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


18:29:29  INFO      Collection 'fia_regulations_minilm_v1' already exists — skipping creation


  -> 'fia_regulations_minilm_v1' already populated with 2279 chunks; skipping rebuild.
fia_regulations_minilm_v1 ready (2279 points)


In [9]:
# Build BGE chunk-256 collection (chunk 256 / overlap 32, 1024-dim)
print(f"\nBuilding {BGE_CHUNK256_COLLECTION}...")
bge_encoder = SentenceTransformer(PROD_MODEL)
n_bge_256 = _build_collection(
    client,
    name=BGE_CHUNK256_COLLECTION,
    encoder=bge_encoder,
    vector_size=1024,
    chunk_size=256,
    chunk_overlap=32,
    docs_dir=DOCS_DIR,
)
print(f"{BGE_CHUNK256_COLLECTION} ready ({n_bge_256} points)")


18:29:29  INFO      Use pytorch device_name: cuda:0


18:29:29  INFO      Load pretrained SentenceTransformer: BAAI/bge-m3


18:29:30  INFO      HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


18:29:30  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"



Building fia_regulations_bge_chunk256_v1...


...

18:29:35  INFO      HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


18:29:35  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


18:29:35  INFO      HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"


18:29:36  INFO      Collection 'fia_regulations_bge_chunk256_v1' already exists — skipping creation


  -> 'fia_regulations_bge_chunk256_v1' already populated with 4558 chunks; skipping rebuild.
fia_regulations_bge_chunk256_v1 ready (4558 points)


## Helpers de evaluación

Una recuperación se considera **correcta** si se cumplen las dos condiciones:

1. **Match de artículo**: el número de artículo esperado aparece en el campo
   `article` del payload del chunk **o** como substring del propio texto del
   chunk. La doble vía es necesaria porque la regex de extracción de
   artículo en `scripts/build_rag_index.py` (`_ARTICLE_RE`) captura solo la
   primera referencia "Article X.Y" del chunk, que con frecuencia es una
   referencia cruzada (por ejemplo "according to Article 30.1a)") y no el
   artículo del que el chunk realmente forma parte. Aceptar también el
   texto del chunk evita penalizar al retriever por una limitación de
   metadatos del pipeline de indexación.
2. **Match de keyword**: al menos una de las `expected_keywords` aparece
   como substring literal (case-insensitive) en el `text` del chunk.

La normalización del artículo elimina la palabra `Article`, espacios y
mayúsculas, de modo que `Article 30.2`, `30.2` y `ARTICLE 30.2` se
consideran equivalentes.

In [10]:
def _normalise_article(value: str) -> str:
    """Normalise an article reference for fuzzy comparison.

    Lowercases, strips whitespace, removes the literal word "article",
    and collapses internal whitespace. Returns just the numeric portion
    (e.g. "30.2") so that "Article 30.2" matches "30.2" and a payload
    field of "  ARTICLE  30.2  " matches both.
    """
    s = (value or "").lower().replace("article", "").strip()
    return " ".join(s.split())


def is_correct(chunk_payload: dict, ground_truth: dict) -> bool:
    """Return True when a retrieved chunk satisfies article + keyword match.

    Both criteria must hold:
    - article match: the ground-truth article number is present in the
      normalised payload article string OR in the chunk text. The double
      check compensates for the indexing-time regex that occasionally
      tags a chunk with a cross-referenced article instead of the article
      it actually belongs to.
    - keyword match: at least one expected_keyword is a case-insensitive
      substring of the chunk text.
    """
    expected_article = _normalise_article(ground_truth.get("article", ""))
    payload_article = _normalise_article(chunk_payload.get("article", ""))
    text = (chunk_payload.get("text", "") or "")

    article_in_payload = bool(expected_article) and (expected_article in payload_article)
    # Match the article number against the chunk text (handles both "30.2" and
    # "Article 30.2" patterns inside the body). The expected_article is already
    # normalised to the digits-and-dot form, e.g. "30.2".
    article_in_text = bool(expected_article) and (expected_article in text)
    article_ok = article_in_payload or article_in_text

    keywords = ground_truth.get("expected_keywords", [])
    text_lower = text.lower()
    keyword_ok = any(kw.lower() in text_lower for kw in keywords)

    return article_ok and keyword_ok


def is_content_correct(chunk_payload: dict, ground_truth: dict) -> bool:
    """Looser, content-only correctness check used as a complementary metric.

    Returns True when at least one expected_keyword is a substring of the
    chunk text, ignoring the article tag entirely. This isolates the
    retrieval quality of the embedding from the noise introduced by the
    article-extraction regex during indexing. The strict ``is_correct``
    metric remains the headline number; this one is reported alongside
    so the thesis can quantify how much of the strict-metric loss is
    attributable to article tagging rather than to the embedding.
    """
    keywords = ground_truth.get("expected_keywords", [])
    text_lower = (chunk_payload.get("text", "") or "").lower()
    return any(kw.lower() in text_lower for kw in keywords)


In [11]:
def evaluate_retriever(
    client: QdrantClient,
    collection: str,
    encoder: SentenceTransformer,
    queries: list[dict],
    *,
    k_max: int = 10,
) -> pd.DataFrame:
    """Run every query against `collection` and return per-query metrics.

    For each query we encode the question once (timed with `time.perf_counter`),
    run a single `query_points` call with `limit=k_max`, then compute
    Precision@1/3/5 and reciprocal rank from the ranked hit list. The
    encoder is passed in explicitly so the caller controls model loading
    (avoids reloading BGE-M3 inside the loop).
    """
    rows: list[dict] = []
    for q in queries:
        ground_truth = q["ground_truth"]

        t0 = time.perf_counter()
        vector = encoder.encode(q["query"], normalize_embeddings=True).tolist()
        response = client.query_points(
            collection_name=collection,
            query=vector,
            limit=k_max,
            with_payload=True,
        )
        elapsed_ms = (time.perf_counter() - t0) * 1000.0

        hits = [is_correct(hit.payload, ground_truth) for hit in response.points]
        content_hits = [is_content_correct(hit.payload, ground_truth) for hit in response.points]
        # Pad to k_max so positional checks are always defined
        while len(hits) < k_max:
            hits.append(False)
        while len(content_hits) < k_max:
            content_hits.append(False)

        precision_at_1 = float(any(hits[:1]))
        precision_at_3 = float(any(hits[:3]))
        precision_at_5 = float(any(hits[:5]))

        content_p_at_5 = float(any(content_hits[:5]))

        rr = 0.0
        for rank, ok in enumerate(hits, start=1):
            if ok:
                rr = 1.0 / rank
                break

        rows.append(
            {
                "query_id": q["id"],
                "category": q["category"],
                "year": q["year"],
                "precision_at_1": precision_at_1,
                "precision_at_3": precision_at_3,
                "precision_at_5": precision_at_5,
                "content_p_at_5": content_p_at_5,
                "rr": rr,
                "latency_ms": elapsed_ms,
            }
        )
    return pd.DataFrame(rows)


## Ejecución de las tres configuraciones

Cada configuración se ejecuta con su propio encoder (los dos encoders BGE-M3
para baseline y chunk-256 son la misma instancia de modelo - se reutiliza
para ahorrar memoria) y se mide en el orden producción → MiniLM →
chunk-256. La latencia incluye la codificación de la query y la búsqueda
en Qdrant; no incluye el postproceso de `is_correct` porque ese coste sería
trivial frente a una llamada del LLM downstream.

In [12]:
# Production retriever (BGE-M3, chunk 512)
print("Evaluating production baseline (BGE-M3 1024d, chunk 512)...")
df_prod = evaluate_retriever(client, PROD_COLLECTION, bge_encoder, QUERIES, k_max=10)
df_prod.head()


Evaluating production baseline (BGE-M3 1024d, chunk 512)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.36it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.36it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 39.93it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 42.21it/s]

,query_id,category,year,precision_at_1,precision_at_3,precision_at_5,content_p_at_5,rr,latency_ms
0,Q01,tyre_allocation,2025,0.0,0.0,0.0,1.0,0.166667,244.2943
1,Q02,tyre_allocation,2023,0.0,0.0,0.0,1.0,0.142857,34.4468
2,Q03,tyre_allocation,2025,0.0,0.0,0.0,0.0,0.000000,34.5700
3,Q04,tyre_allocation,2025,0.0,0.0,0.0,1.0,0.000000,33.0488
4,Q05,pit_stops,2025,1.0,1.0,1.0,1.0,1.000000,33.4086


In [13]:
# MiniLM baseline
print("Evaluating MiniLM-L6-v2 (384d, chunk 512)...")
df_minilm = evaluate_retriever(client, MINILM_COLLECTION, minilm_encoder, QUERIES, k_max=10)
df_minilm.head()


Evaluating MiniLM-L6-v2 (384d, chunk 512)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.56it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 177.63it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 100.01it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 200.01it/s]

,query_id,category,year,precision_at_1,precision_at_3,precision_at_5,content_p_at_5,rr,latency_ms
0,Q01,tyre_allocation,2025,0.0,0.0,0.0,1.0,0.0,26.1203
1,Q02,tyre_allocation,2023,0.0,0.0,0.0,0.0,0.0,24.2626
2,Q03,tyre_allocation,2025,0.0,0.0,0.0,0.0,0.0,26.2270
3,Q04,tyre_allocation,2025,0.0,0.0,0.0,0.0,0.0,17.5492
4,Q05,pit_stops,2025,0.0,1.0,1.0,1.0,0.5,16.5979


In [14]:
# BGE chunk-256 variant
print("Evaluating BGE-M3 chunk-256 variant...")
df_bge_chunk256 = evaluate_retriever(client, BGE_CHUNK256_COLLECTION, bge_encoder, QUERIES, k_max=10)
df_bge_chunk256.head()


Evaluating BGE-M3 chunk-256 variant...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 52.10it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 44.34it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 42.14it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 44.79it/s]

,query_id,category,year,precision_at_1,precision_at_3,precision_at_5,content_p_at_5,rr,latency_ms
0,Q01,tyre_allocation,2025,0.0,0.0,0.0,1.0,0.1,39.8060
1,Q02,tyre_allocation,2023,0.0,0.0,0.0,1.0,0.0,45.2966
2,Q03,tyre_allocation,2025,0.0,0.0,0.0,0.0,0.0,53.1291
3,Q04,tyre_allocation,2025,0.0,0.0,0.0,1.0,0.0,39.2400
4,Q05,pit_stops,2025,1.0,1.0,1.0,1.0,1.0,43.4480


## Tabla resumen comparativa

Una fila por configuración. Las columnas Precision@1/3/5 y MRR se promedian
sobre las 15 queries; las latencias se reportan como percentiles 50 y 95
(ms). La P95 es la métrica relevante para el SLA del agente porque captura
la cola de la distribución, no la mediana.

In [15]:
def _summarise(df: pd.DataFrame, name: str) -> dict:
    return {
        "config": name,
        "precision_at_1": df["precision_at_1"].mean(),
        "precision_at_3": df["precision_at_3"].mean(),
        "precision_at_5": df["precision_at_5"].mean(),
        "content_p_at_5": df["content_p_at_5"].mean(),
        "mrr": df["rr"].mean(),
        "latency_p50_ms": df["latency_ms"].quantile(0.50),
        "latency_p95_ms": df["latency_ms"].quantile(0.95),
    }


summary_rows = [
    _summarise(df_prod, "BGE-M3 chunk 512 (production)"),
    _summarise(df_minilm, "MiniLM-L6-v2 chunk 512"),
    _summarise(df_bge_chunk256, "BGE-M3 chunk 256"),
]
summary_df = pd.DataFrame(summary_rows)
summary_df_round = summary_df.copy()
for col in ("precision_at_1", "precision_at_3", "precision_at_5", "content_p_at_5", "mrr"):
    summary_df_round[col] = summary_df_round[col].round(3)
for col in ("latency_p50_ms", "latency_p95_ms"):
    summary_df_round[col] = summary_df_round[col].round(1)
summary_df_round


,config,precision_at_1,precision_at_3,precision_at_5,content_p_at_5,mrr,latency_p50_ms,latency_p95_ms
0,BGE-M3 chunk 512 (production),0.133,0.133,0.200,0.800,0.191,34.6,107.8
1,MiniLM-L6-v2 chunk 512,0.000,0.067,0.067,0.267,0.033,17.5,26.2
2,BGE-M3 chunk 256,0.067,0.067,0.067,0.800,0.093,45.3,51.6


## Discusión de resultados

La tabla anterior tiene seis métricas por configuración:

- **Precision@1/3/5 (estricta)**: el chunk recuperado debe contener el
  artículo correcto (en el campo `article` del payload o en el cuerpo del
  texto del chunk) **y** al menos una keyword esperada.
- **Content P@5**: relaja la condición y solo exige keyword en el texto.
  Esta métrica aísla la calidad del embedding del ruido del article-tag.
- **MRR / Latencia P50/P95**: complementan la calidad y el coste.

La diferencia entre `Precision@5 (estricta)` y `Content P@5` mide el coste
de la regex `_ARTICLE_RE` que tagea cada chunk con la primera referencia
`Article X.Y` que encuentra; en chunks cuyo cuerpo cita al artículo
vecino, la regex captura ese cross-reference y no el artículo del que el
chunk forma parte. Esta limitación es conocida y fuera de alcance del
benchmark; el thesis puede reportar las dos métricas para ser transparente.

Además, la tabla cuantifica tres comparaciones para el capítulo 5:

1. **BGE-M3 vs MiniLM-L6-v2 (mismo chunking).** BGE-M3 es un modelo
   multilingüe entrenado con un contraste a gran escala y MTEB ~67;
   MiniLM-L6-v2 es ~6× más pequeño (384d frente a 1024d) y entrenado solo
   en inglés. La diferencia de Precision@k cuantifica cuánto pierde el
   pipeline al sustituir el modelo de producción por una alternativa
   ligera; la diferencia de latencia cuantifica el ahorro CPU
   correspondiente.

2. **BGE-M3 chunk 512 vs chunk 256 (mismo modelo).** Chunks más finos
   (256 caracteres ≈ medio artículo) dan mayor granularidad pero también
   más probabilidad de partir un artículo en dos fragmentos, dejando una
   parte del contenido fuera del top-k. Comparar Precision@5 entre las dos
   configuraciones es la forma directa de medir este trade-off.

3. **Latencia P50 / P95.** El retriever se llama una vez por turno del
   orquestador; una P95 < 100 ms hace que el RAG no sea el cuello de
   botella frente al LLM (varios cientos de ms incluso con un modelo
   local).

### Casos de fallo típicos

- **Chunking demasiado fino**: artículos como `30.2` (que ocupa más de
  500 caracteres en el PDF) se parten en dos chunks; el chunk con la
  cláusula numérica concreta ("thirteen (13) sets") puede quedar fuera
  del top-k aunque el artículo sí esté representado por su header.
- **Query ambigua**: queries que mencionan a la vez "DRS" y "Safety
  Car" (Q14, Q15) compiten contra los chunks de las propias secciones
  de Safety Car y Virtual Safety Car, que también mencionan ambos
  términos.
- **Año-equivocado**: el retriever no filtra por temporada en
  producción, así que para una query 2023 puede devolver un chunk 2025
  con `article` correcto pero contenido distinto. Las keywords
  literales suelen capturar este fallo (las cantidades cambian entre
  años) pero no en todos los casos.

### Hallazgos por categoría

El DataFrame `df_prod` (y sus equivalentes para las otras dos configs)
permite romper las métricas por categoría. La celda siguiente lo expone
para la baseline de producción - una categoría con Precision@5 < 0,5
indica un punto débil concreto del retriever que el capítulo 5 puede
discutir cualitativamente.

In [16]:
def _per_category(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("category")
        [["precision_at_1", "precision_at_3", "precision_at_5", "rr"]]
        .mean()
        .round(3)
    )


print("Per-category metrics - production baseline (BGE-M3 chunk 512):")
_per_category(df_prod)


Per-category metrics - production baseline (BGE-M3 chunk 512):


,precision_at_1,precision_at_3,precision_at_5,rr
category,,,,
drs,0.000,0.000,0.000,0.000
flags_penalties,0.000,0.000,0.000,0.056
pit_stops,0.333,0.333,0.333,0.333
safety_car,0.333,0.333,0.667,0.464
tyre_allocation,0.000,0.000,0.000,0.077


In [17]:
print("Per-category metrics - MiniLM-L6-v2 chunk 512:")
_per_category(df_minilm)


Per-category metrics - MiniLM-L6-v2 chunk 512:


,precision_at_1,precision_at_3,precision_at_5,rr
category,,,,
drs,0.0,0.000,0.000,0.000
flags_penalties,0.0,0.000,0.000,0.000
pit_stops,0.0,0.333,0.333,0.167
safety_car,0.0,0.000,0.000,0.000
tyre_allocation,0.0,0.000,0.000,0.000


In [18]:
print("Per-category metrics - BGE-M3 chunk 256:")
_per_category(df_bge_chunk256)


Per-category metrics - BGE-M3 chunk 256:


,precision_at_1,precision_at_3,precision_at_5,rr
category,,,,
drs,0.000,0.000,0.000,0.000
flags_penalties,0.000,0.000,0.000,0.000
pit_stops,0.333,0.333,0.333,0.333
safety_car,0.000,0.000,0.000,0.097
tyre_allocation,0.000,0.000,0.000,0.025


## Exportación del reporte

Generamos `data/rag_eval/results_v1.md` con el resumen comparativo y la
discusión narrativa, listo para copiar al capítulo 5 de la memoria. El
fichero se sobrescribe en cada Restart-Run-All.

In [19]:
def _format_summary_row(row: pd.Series) -> str:
    return (
        f"| {row['config']} "
        f"| {row['precision_at_1']:.3f} "
        f"| {row['precision_at_3']:.3f} "
        f"| {row['precision_at_5']:.3f} "
        f"| {row['content_p_at_5']:.3f} "
        f"| {row['mrr']:.3f} "
        f"| {row['latency_p50_ms']:.1f} "
        f"| {row['latency_p95_ms']:.1f} |"
    )


def export_report(summary: pd.DataFrame, queries: list[dict], path: Path) -> None:
    """Write a self-contained Markdown report with the summary table + discussion."""
    n_queries = len(queries)
    categories = sorted({q["category"] for q in queries})

    lines: list[str] = []
    lines.append("# Resultados del benchmark cuantitativo del RAG Agent (N30)")
    lines.append("")
    lines.append(
        f"Set de evaluación: {n_queries} queries en español sobre el "
        f"reglamento deportivo FIA 2023-2025, distribuidas en "
        f"{len(categories)} categorías ({', '.join(categories)}). Cada query "
        "lleva ground truth manual verificada como substring literal del PDF "
        "correspondiente."
    )
    lines.append("")
    lines.append(
        "**Métricas reportadas.** `Precision@k (estricta)` exige match de "
        "artículo (en el payload o en el texto del chunk) **y** match de "
        "keyword en el texto. `Content P@5` relaja la condición y exige solo "
        "match de keyword, aislando así la calidad del embedding del ruido "
        "del article-tagging del indexador. Reportar ambas columnas es la "
        "forma honesta de presentar el resultado: la métrica estricta refleja "
        "lo que el agente realmente cita, la de contenido refleja lo que el "
        "retriever realmente encuentra."
    )
    lines.append("")
    lines.append("## Tabla resumen comparativa")
    lines.append("")
    lines.append(
        "| Configuración | Precision@1 | Precision@3 | Precision@5 | "
        "Content P@5 | MRR | Latencia P50 (ms) | Latencia P95 (ms) |"
    )
    lines.append(
        "|---|---|---|---|---|---|---|---|"
    )
    for _, row in summary.iterrows():
        lines.append(_format_summary_row(row))
    lines.append("")
    lines.append("## Discusión")
    lines.append("")
    lines.append(
        "**Precision@5 estricta vs Content P@5.** La métrica estricta exige "
        "match de artículo (en payload o en texto del chunk) y match de "
        "keyword. La métrica `content_p_at_5` ignora el campo `article` y "
        "exige solo presencia de keyword en el texto, aislando así la "
        "calidad pura del embedding. La diferencia entre las dos columnas "
        "cuantifica el coste de la regex `_ARTICLE_RE` de "
        "`scripts/build_rag_index.py`, que tagea cada chunk con la primera "
        "referencia `Article X.Y` que encuentra - referencia que en muchos "
        "chunks corresponde a una cita cruzada (por ejemplo `Article "
        "30.1a)`) y no al artículo del que el chunk realmente forma parte. "
        "Esta degradación es una limitación conocida del pipeline de "
        "indexación; mitigarla requiere un post-proceso del payload "
        "(enriquecer con la heading más cercana del documento) que está "
        "fuera de alcance de este benchmark."
    )
    lines.append("")
    lines.append(
        "**BGE-M3 vs MiniLM-L6-v2.** BGE-M3 es un modelo multilingüe de 1024d "
        "con MTEB ~67 entrenado con contraste masivo; MiniLM-L6-v2 es ~6x más "
        "pequeño (384d) y monolingüe en inglés. La diferencia de Precision@k "
        "entre filas 1 y 2 de la tabla cuantifica el coste en calidad de "
        "sustituir el modelo de producción por una alternativa ligera, y la "
        "diferencia de latencia cuantifica el ahorro CPU correspondiente."
    )
    lines.append("")
    lines.append(
        "**Chunk 512 vs chunk 256 con BGE-M3.** Chunks más finos dan mayor "
        "granularidad de recuperación pero aumentan la probabilidad de partir "
        "un artículo en dos fragmentos. La comparación filas 1 y 3 de la tabla "
        "mide ese trade-off de forma directa: si Precision@5 cae al pasar a "
        "chunk 256, el chunking fino está dejando contenido relevante fuera "
        "del top-k."
    )
    lines.append("")
    lines.append(
        "**Latencia P50 / P95.** El retriever se llama una vez por turno del "
        "orquestador. Una P95 por debajo de 100 ms hace que el RAG no sea el "
        "cuello de botella frente a la llamada al LLM (varios cientos de ms "
        "incluso con un modelo local), de modo que el coste de la fase de "
        "recuperación es absorbible dentro del SLA del agente."
    )
    lines.append("")
    lines.append("### Casos de fallo típicos")
    lines.append("")
    lines.append(
        "- **Chunking demasiado fino**: artículos como `30.2`, que ocupan más "
        "de 500 caracteres en el PDF, se parten en dos chunks. El chunk con "
        "la cláusula numérica concreta (por ejemplo `thirteen (13) sets`) "
        "puede quedar fuera del top-k aunque el artículo sí esté representado "
        "por su header."
    )
    lines.append(
        "- **Query ambigua**: queries que mencionan simultáneamente DRS y "
        "Safety Car (Q14, Q15) compiten contra los chunks de las secciones "
        "55 (Safety Car) y 56 (Virtual Safety Car), que también mencionan "
        "ambos términos."
    )
    lines.append(
        "- **Año equivocado**: el retriever no filtra por temporada, así que "
        "para una query 2023 puede devolver un chunk 2025 con artículo "
        "correcto pero contenido distinto. Las keywords literales suelen "
        "capturar este fallo cuando los valores numéricos cambian entre años "
        "(intermediates: 4 sets en 2023 vs 5 sets en 2025; DRS habilitado "
        "tras 2 vueltas en 2023 vs 1 vuelta en 2025) pero no en todos los "
        "casos."
    )
    lines.append("")
    lines.append("### Reproducibilidad")
    lines.append("")
    lines.append(
        "El cuaderno `notebooks/agents/N30B_rag_benchmark.ipynb` reconstruye "
        "el set y las dos colecciones nuevas de Qdrant de forma idempotente: "
        "Restart Kernel → Run All las salta si ya existen. El JSON con las "
        "queries vive en `data/rag_eval/queries_v1.json` y la verificación "
        "literal de keywords es responsabilidad del autor del set, no del "
        "cuaderno - cualquier ampliación futura debe pasar por el mismo "
        "filtro contra los PDFs originales."
    )
    lines.append("")

    path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Wrote {path} ({len(lines)} lines)")


REPORT_PATH = REPO_ROOT / "data" / "rag_eval" / "results_v1.md"
export_report(summary_df, QUERIES, REPORT_PATH)


Wrote C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\rag_eval\results_v1.md (34 lines)


## Cierre

La métrica de validación es `Content P@5` de la baseline de producción:
debe situarse cómodamente por encima de 0,5 para que las comparaciones
laterales (MiniLM, chunk 256) sean significativas. La métrica estricta
`Precision@5` viene degradada estructuralmente por el article-tagging del
indexador y no debe usarse como umbral de éxito sin tener en cuenta esa
limitación; reportar ambas en la memoria es lo más honesto.